In [1]:
# for working part
import numpy as np
import pandas as pd



#  for ml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score

In [2]:
# reading the parquet file created in notebook 2

lead_conversion_data = pd.read_parquet("C:\\Users\\hp5cd\\OneDrive\\Desktop\\Python\\lead-conversion-prediction\\data\\processed_data.parquet")

In [3]:
lead_conversion_data.columns

Index(['owner', 'lead_source', 'profile', 'total_duration', 'call_count',
       'distinct_call_days', 'connected_call_count', 'missed_call_count',
       'inbound_call_count', 'outbound_call_count', 'converted',
       'assigned_month', 'assigned_year', 'followup_done', 'average_duration',
       'connection_rate', 'miss_rate', 'average_call_per_day',
       'average_duration_per_day', 'inbound_outbound_ratio',
       'time_taken_for_first_touch', 'call_span_days', 'call_frequency'],
      dtype='object')

In [4]:
# hume neeche model train krne mein errors aaye toh humne ye kiya hai again aur phir se re run krne ja rhe hai codes

lead_conversion_data['inbound_outbound_ratio'] = lead_conversion_data['inbound_outbound_ratio'].replace([np.inf,-np.inf],np.nan).fillna(0)

lead_conversion_data.fillna(0,inplace=True)

lead_conversion_data['profile'] = lead_conversion_data['profile'].replace(0,"unknown_profile")

In [ ]:
# seperate input and output columns


x3 = lead_conversion_data[['owner', 'lead_source', 'profile', 'total_duration', 'call_count',
       'distinct_call_days', 'connected_call_count', 'missed_call_count',
       'inbound_call_count', 'outbound_call_count',
       'assigned_month', 'assigned_year', 'followup_done', 'average_duration',
       'connection_rate', 'miss_rate', 'average_call_per_day',
       'average_duration_per_day', 'inbound_outbound_ratio',
       'time_taken_for_first_touch', 'call_span_days', 'call_frequency']]



y = lead_conversion_data[['converted']]

In [ ]:
## stratify is used to keep the distribution of output variables same in test and train split
## test size 0.2 means 80% training split of data and 20% will be in test split

x3_train, x3_test, y3_train, y3_test = train_test_split(x3,y,random_state=42,stratify=y,test_size=0.2)

In [ ]:
# seperating three type of columns from input x


categorical_x3 = ['owner', 'lead_source', 'profile','assigned_month',
                   'assigned_year']

numerical_x3 = ['total_duration', 'call_count','distinct_call_days',
                 'connected_call_count', 'missed_call_count',
                 'inbound_call_count', 'outbound_call_count','average_duration',
                 'connection_rate', 'miss_rate', 'average_call_per_day',
                 'average_duration_per_day', 'inbound_outbound_ratio',
                 'time_taken_for_first_touch', 'call_span_days', 'call_frequency']

binary_x3 = ['followup_done']


In [ ]:
# preprocessing code line


preprocessor3 = ColumnTransformer(transformers=[('cat',OneHotEncoder(handle_unknown='ignore'),categorical_x3),
                                               ('num',StandardScaler(),numerical_x3),
                                               ('binary','passthrough',binary_x3)])



# models to train with their parameters


models = {'LR':LogisticRegression(max_iter=1000),
          'RF':RandomForestClassifier(n_estimators=20,max_depth=10,n_jobs=-1,random_state=42),
          'XGB': XGBClassifier(n_estimators=50,max_depth=5,learning_rate=0.1,n_jobs=-1,random_state=42)}




# loop for training and testing and scores model one by one

for name, model in models.items():
    pipeline = Pipeline([('preprocess',preprocessor3),
                         ('model',model)])
    
    pipeline.fit(x3_train,y3_train)

    y3_pred = pipeline.predict(x3_test)
    y3_pred_prob = pipeline.predict_proba(x3_test)[:,1]


    print("model", name)

    print(" ")
    print("accuracy_score: ", accuracy_score(y3_test,y3_pred))
    print("confusion_matrix: ", confusion_matrix(y3_test,y3_pred))
    print("precision_score: ", precision_score(y3_test,y3_pred))
    print("recall_score: ", recall_score(y3_test,y3_pred))
    print("f1_score: ", f1_score(y3_test,y3_pred))
    print("roc_auc_score: ", roc_auc_score(y3_test,y3_pred_prob))

    print(" ")


c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


model LR
 
accuracy_score:  0.7243357142857143
confusion_matrix:  [[78408 16108]
 [22485 22999]]
precision_score:  0.588104431431713
recall_score:  0.5056503385805998
f1_score:  0.5437694317362367
roc_auc_score:  0.7903664963467547
 


c:\Users\hp5cd\OneDrive\Desktop\Python\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


model RF
 
accuracy_score:  0.7031071428571428
confusion_matrix:  [[88447  6069]
 [35496  9988]]
precision_score:  0.6220340038612443
recall_score:  0.2195937032802744
f1_score:  0.3245966103898214
roc_auc_score:  0.7718524531466935
 
model XGB
 
accuracy_score:  0.7227142857142858
confusion_matrix:  [[77852 16664]
 [22156 23328]]
precision_score:  0.5833166633326665
recall_score:  0.5128836513938968
f1_score:  0.5458374280499789
roc_auc_score:  0.7883890307175242
 


In [ ]:
# checking where there is infinity value as first time there was a infinity value type error while training

np.isinf(x3_train[numerical_x3]).sum()

total_duration                   0
call_count                       0
distinct_call_days               0
connected_call_count             0
missed_call_count                0
inbound_call_count               0
outbound_call_count              0
average_duration                 0
connection_rate                  0
miss_rate                        0
average_call_per_day             0
average_duration_per_day         0
inbound_outbound_ratio        5347
time_taken_for_first_touch       0
call_span_days                   0
call_frequency                   0
dtype: int64